# Optimizers — companion notebook

Companion for [`optimizers.md`](./optimizers.md). We compare SGD, Momentum, RMSProp, and Adam on the same 2D objective and visualize their trajectories.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Ill-conditioned quadratic: f(x,y) = x^2 + 10 y^2
def f(p): return p[0]**2 + 10*p[1]**2
def grad(p): return np.array([2*p[0], 20*p[1]])

def run(opt, x0, steps=120, **kwargs):
    x = np.array(x0, dtype=float); path = [x.copy()]
    state = {}
    for t in range(1, steps+1):
        g = grad(x)
        x = opt(x, g, t, state, **kwargs); path.append(x.copy())
    return np.array(path)

def sgd(x, g, t, s, lr=0.04): return x - lr*g
def momentum(x, g, t, s, lr=0.04, beta=0.9):
    s['m'] = beta*s.get('m', 0) + g; return x - lr*s['m']
def rmsprop(x, g, t, s, lr=0.1, rho=0.9, eps=1e-8):
    s['v'] = rho*s.get('v', 0) + (1-rho)*g**2
    return x - lr*g/(np.sqrt(s['v'])+eps)
def adam(x, g, t, s, lr=0.2, b1=0.9, b2=0.999, eps=1e-8):
    s['m'] = b1*s.get('m', 0) + (1-b1)*g
    s['v'] = b2*s.get('v', 0) + (1-b2)*g**2
    mh = s['m']/(1 - b1**t); vh = s['v']/(1 - b2**t)
    return x - lr*mh/(np.sqrt(vh)+eps)

x0 = np.array([7.0, 1.5])
paths = {'SGD': run(sgd, x0), 'Momentum': run(momentum, x0), 'RMSProp': run(rmsprop, x0), 'Adam': run(adam, x0)}

In [ ]:
gx, gy = np.meshgrid(np.linspace(-8, 8, 400), np.linspace(-2, 2, 400))
Z = gx**2 + 10*gy**2
fig, ax = plt.subplots(figsize=(10, 4))
ax.contour(gx, gy, np.log(Z+0.1), levels=25, cmap='Greys', alpha=0.6)
for name, p in paths.items():
    ax.plot(p[:,0], p[:,1], '.-', label=name, ms=4)
ax.scatter([0],[0], color='gold', marker='*', s=150, label='minimum', zorder=5)
ax.set_title('Trajectories on f(x,y) = x^2 + 10 y^2'); ax.legend(); ax.set_aspect('equal')
plt.show()